# Track 6 — GPU-Native Fokker–Planck Closure Surrogate for a Lid-Driven Cavity

**Advanced instructor-approved research project**

This project reproduces the central workflow of the instructor's machine-learning/Fokker–Planck cavity study at a reduced educational scale:

1. run the exact cubic-FP closure to generate labeled data;
2. train a neural network that maps 16 low-order moments to 9 closure coefficients;
3. export the trained weights in a format that the CuPy particle solver can evaluate directly on the GPU;
4. insert the neural closure into a complete held-out cavity simulation;
5. compare exact-FP and ML-FP fields, high-order diagnostics, physical checks, and runtime.

The associated conference presentation is by **Ehsan Roohi and Francis Padilla**. This notebook is deliberately more guided than the other project tracks because the kinetic solver and closure equations are research-level code.

> **Central question:** Does a q-weighted neural closure improve prediction of the heat-flux coefficient block on complete held-out cavity conditions without unacceptable degradation of the stress-relaxation block, and does the selected model remain accurate and stable when deployed inside the full particle-FP cavity solver?


<!-- MIE690A enriched learner edition v2 -->

## How to learn from this notebook

This is a guided computational laboratory, not a script to execute without reading. For every numbered stage:

1. read the physical question and write a prediction;
2. inspect the inputs, outputs, units, and split before running code;
3. run the cell and check assertions/warnings;
4. compare with the stated baseline or physical diagnostic; and
5. write one or two sentences explaining what the result does **and does not** establish.

Use **Restart and Run All** before treating any output as final. Hidden state from out-of-order execution is a reproducibility failure.

### Evidence contract

Keep four kinds of evidence separate:

- **numerical evidence:** residuals, accepted cases, data hashes, grid/time/particle budgets;
- **statistical evidence:** losses, relative errors, variability across seeds/cases;
- **physical evidence:** centerlines, walls, divergence, vortex structure, positivity, moments;
- **computational evidence:** runtime, memory, saved configuration, and machine-readable metrics.

A claim is only as strong as the weakest relevant layer.


## Prerequisite gate and scope

Proceed only if you can already explain case-wise splitting, standardized multi-output loss, GPU/CPU timing synchronization, and the difference between offline and a-posteriori validation. Track 6 supplies the kinetic solver and wrapper scripts; the scientific work is the controlled closure experiment, not rewriting the research solver.

The reduced fast mode verifies software plumbing. It cannot support production speedup, convergence, or general rarefaction claims.


In [ ]:
# Repository/Colab bootstrap for Track 6.
from pathlib import Path
import os, sys

def _find_track6_dir(start=Path.cwd()):
    for base in [start, *start.parents]:
        candidate = base / "advanced" / "fp_closure"
        if (candidate / "fp_cavity_reference.py").exists():
            return candidate
    if (start / "fp_cavity_reference.py").exists():
        return start
    raise FileNotFoundError(
        "Track-6 scripts not found. Run from the course repository or upload all files "
        "from advanced/fp_closure into the Colab Files pane."
    )

TRACK6_DIR = _find_track6_dir()
sys.path.insert(0, str(TRACK6_DIR))
os.chdir(TRACK6_DIR)
print("Track-6 working directory:", TRACK6_DIR)


## Learning outcomes

By the end of the project you should be able to:

- explain what the exact 9×9 cubic-FP closure calculation does;
- distinguish the 16 low-order **input features** from the 9 exact **target coefficients**;
- prevent leakage by separating complete simulation conditions;
- compare a uniform coefficient loss with a q-weighted loss;
- export a trained Keras network to plain arrays for native CuPy inference;
- perform both **offline coefficient validation** and **closed-loop cavity validation**;
- separate coefficient-fit error, closed-loop field error, statistical particle noise, numerical bias, and out-of-range physics;
- state a bounded conclusion suitable for a conference presentation.


## Scientific workflow

The exact cubic-FP solver computes high-order moments and solves a local 9×9 linear system in every cell and time step. The neural network replaces that deterministic closure calculation:

\[
\underbrace{\mathbf x}_{16\ \text{low-order features}}
\quad\longmapsto\quad
\underbrace{\mathbf y}_{9\ \text{closure coefficients}}.
\]

The input vector is

\[
\mathbf x=(\rho,T,U_x,U_y,U_z,\Pi_{xx},\Pi_{xy},\Pi_{xz},\Pi_{yy},\Pi_{yz},\Pi_{zz},q_x,q_y,q_z,DM2,\nu).
\]

The output vector is

\[
\mathbf y=(C_{xx},C_{xy},C_{xz},C_{yy},C_{yz},C_{zz},\Gamma_x,\Gamma_y,\Gamma_z).
\]

The six \(C_{ij}\) coefficients control stress relaxation; the three \(\Gamma_i\) coefficients control heat-flux relaxation. The reference solver obtains them from the exact local closure. The neural surrogate learns the same map only on the distribution manifold represented by the training simulations.


### Important implementation and timing notes

- The **nine learned outputs** are stored in `coeffs['A']` (six \(C_{ij}\) values) and `coeffs['B']` (three \(\Gamma_i\) values). The separate legacy scalar `coeffs['C']` is zero in **both** the exact and ML paths of the supplied reference solver. Therefore, `coeffs['C'].fill(0)` is not an ML-only omission.
- The measured speedup combines **two** savings: replacing FULL high-order moments by the LITE low-order pass, and replacing closure assembly / the local \(9\times9\) solve by a native DNN forward pass. Do not report it as the arithmetic speed of the network alone.
- Track 6 requires an NVIDIA GPU. If CuPy/CUDA is unavailable in Colab, use **Runtime -> Change runtime type -> GPU**, restart, and run the CuPy installation cell.


## Files in this track

- `fp_cavity_reference.py` — supplied research solver; do not rewrite its closure equations.
- `fp_project_utils.py` — bounded configuration and feature/target definitions.
- `generate_fp_cavity_dataset.py` — exact-FP data generation.
- `train_fp_closure.py` — uniform or q-weighted TensorFlow training and CuPy export.
- `evaluate_fp_closure.py` — coefficient-level validation on complete conditions.
- `run_fp_cavity_test.py` — exact-FP versus ML-FP closed-loop blind cavity test.
- `analyze_fp_cavity_results.py` — field metrics, centerlines, physical checks, and runtime.

The supplied defaults are reduced educational budgets. They prove the workflow, not publication-level convergence.


## 0. Environment and file check

In [ ]:
from pathlib import Path
import importlib.util, json, os, subprocess, sys

REQUIRED = [
    'fp_cavity_reference.py',
    'fp_project_utils.py',
    'generate_fp_cavity_dataset.py',
    'train_fp_closure.py',
    'evaluate_fp_closure.py',
    'run_fp_cavity_test.py',
    'analyze_fp_cavity_results.py',
]
missing = [f for f in REQUIRED if not Path(f).exists()]
assert not missing, f'Missing files: {missing}'
print('All required Track-6 files are present.')

print('Python:', sys.version)
print('TensorFlow available:', importlib.util.find_spec('tensorflow') is not None)
print('CuPy available:', importlib.util.find_spec('cupy') is not None)


### Colab GPU setup

Use a **GPU runtime**. TensorFlow is normally already installed in Colab. Install CuPy only if the previous cell reports that it is missing. After installing CuPy, restart the runtime once if Colab requests it.


In [ ]:
# Run only when CuPy is missing in a Colab GPU runtime.
# !pip -q install 'cupy-cuda12x>=13,<14'
# !nvidia-smi


## Four nested levels of validation

1. **Data integrity:** finite inputs/targets, named columns, complete-condition provenance.
2. **Offline coefficient accuracy:** errors in the six stress-related `C` and three heat-flux-related `Gamma` outputs.
3. **Closed-loop macroscopic fidelity:** density, velocity, temperature, pressure, and centerlines after recursive deployment.
4. **High-order/stability fidelity:** stress, heat flux, positivity, particle health, and time histories.

Passing one level does not imply passing the next. The blind experimental design must therefore name evidence at all four levels.


## 1. Freeze the experimental design before generating blind data

The required controlled comparison changes only the training loss:

- **Baseline:** uniform standardized coefficient MSE.
- **Modification:** q-weighted standardized coefficient MSE, emphasizing the heat-flux block \(\Gamma_i\).

All of the following remain fixed: training conditions, validation condition, network architecture, optimizer, batch size, random seed, input/output scaling, and stopping rule.

The recommended complete-condition split is:

- training: \(U_{lid}=100,200,400,600\) m/s at nominal \(Kn\approx0.15\);
- validation: \(U_{lid}=500\) m/s;
- blind test: \(U_{lid}=800\) m/s.

Do not generate or inspect the 800 m/s data until the model-selection rule has been written and both models have been compared on the 500 m/s validation condition.


In [ ]:
# ---------------- STUDENT CONFIGURATION CELL ----------------
FAST_MODE = True  # True for a smoke run; set False for the final submitted run.

TRAIN_LID_SPEEDS = [100, 200, 400, 600]
VALIDATION_LID_SPEED = 500
BLIND_LID_SPEED = 800
DENSITY_SCALE = 1.0          # nominal Kn ≈ 0.15 / density_scale
Q_WEIGHT = 6.0
MODEL_SEED = 123

if FAST_MODE:
    DATA_NX = DATA_NY = 16
    DATA_PPC = 40
    DATA_STEPS = 700
    DATA_SAMPLE_START = 350
    DATA_SAMPLE_STRIDE = 25
    TRAIN_EPOCHS = 50
    TEST_NX = TEST_NY = 18
    TEST_PPC = 50
    TEST_STEPS = 1000
    TEST_SAMPLE_START = 600
    TEST_SAMPLE_STRIDE = 10
else:
    DATA_NX = DATA_NY = 20
    DATA_PPC = 80
    DATA_STEPS = 1200
    DATA_SAMPLE_START = 600
    DATA_SAMPLE_STRIDE = 20
    TRAIN_EPOCHS = 120
    TEST_NX = TEST_NY = 24
    TEST_PPC = 100
    TEST_STEPS = 1800
    TEST_SAMPLE_START = 1000
    TEST_SAMPLE_STRIDE = 10

print('Final mode?' , not FAST_MODE)
print('Approximate nominal Kn:', 0.15 / DENSITY_SCALE)


## 2. Generate exact-FP training and validation data

At each sampled step, the reference solver:

1. computes the low-order and high-order particle moments;
2. assembles the exact local 9×9 closure system;
3. solves for the nine coefficients;
4. saves the 16 low-order inputs and 9 exact targets.

The data generator keeps complete physical conditions in separate files. It does not randomly mix validation cells into training.


In [ ]:
import subprocess, sys

def run_command(args):
    print(' '.join(map(str, args)))
    subprocess.run(args, check=True)

common_data_args = [
    '--nx', str(DATA_NX), '--ny', str(DATA_NY),
    '--ppc', str(DATA_PPC),
    '--steps', str(DATA_STEPS),
    '--sample-start', str(DATA_SAMPLE_START),
    '--sample-stride', str(DATA_SAMPLE_STRIDE),
]

run_command([
    sys.executable, 'generate_fp_cavity_dataset.py',
    '--out', 'fp_train_data.npz', '--role', 'train',
    '--lid-speeds', *map(str, TRAIN_LID_SPEEDS),
    '--density-scales', str(DENSITY_SCALE),
    '--seed', '101', *common_data_args,
])

run_command([
    sys.executable, 'generate_fp_cavity_dataset.py',
    '--out', 'fp_validation_data.npz', '--role', 'validation',
    '--lid-speeds', str(VALIDATION_LID_SPEED),
    '--density-scales', str(DENSITY_SCALE),
    '--seed', '201', *common_data_args,
])


## Feature observability and loss weighting

The 16 inputs are low-order local features; the nine targets are closure coefficients computed by the exact local system. A coefficient can be difficult to learn because the available inputs do not uniquely observe the required high-order state. Increasing network capacity cannot repair missing information.

Q-weighting changes which standardized output errors dominate optimization. It does not add information. Inspect feature ranges by physical condition and check whether the blind condition lies outside training support.


## 3. Audit the generated dataset before training

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

train = np.load('fp_train_data.npz', allow_pickle=True)
val = np.load('fp_validation_data.npz', allow_pickle=True)

print('Training:', train['inputs'].shape, train['targets'].shape)
print('Validation:', val['inputs'].shape, val['targets'].shape)
print('Features:', train['feature_names'].tolist())
print('Targets:', train['target_names'].tolist())
print('Training lid speeds:', np.unique(train['U_lid']))
print('Validation lid speeds:', np.unique(val['U_lid']))
print('Any nonfinite values?',
      np.any(~np.isfinite(train['inputs'])), np.any(~np.isfinite(train['targets'])))

fig, ax = plt.subplots(figsize=(6,4))
ax.hist(train['q_norm'], bins=50, alpha=0.7, label='training')
ax.hist(val['q_norm'], bins=50, alpha=0.6, label='validation')
ax.set_xlabel('dimensionless q_norm')
ax.set_ylabel('count')
ax.legend()
plt.show()


### Interpretation prompt

Explain in your notes:

1. Why are the input and output columns standardized using training statistics only?
2. Why is `q_norm` useful for identifying difficult non-equilibrium states?
3. Why is the validation condition a complete simulation rather than a random 10% of the training rows?


## 4. Train the uniform-loss baseline

The baseline minimizes

\[
\mathcal L_{uniform}=\frac{1}{N}\sum_{n=1}^{N}\sum_{j=1}^{9}
(\widehat y_{n,j}^{(s)}-y_{n,j}^{(s)})^2,
\]

where superscript \((s)\) denotes standardized coefficients.


In [ ]:
run_command([
    sys.executable, 'train_fp_closure.py',
    '--train', 'fp_train_data.npz',
    '--val', 'fp_validation_data.npz',
    '--out', 'fp_model_uniform.npz',
    '--loss-mode', 'uniform',
    '--epochs', str(TRAIN_EPOCHS),
    '--hidden', '128', '--batch', '2048',
    '--seed', str(MODEL_SEED),
])


## 5. Train the q-weighted modification

The modified loss is

\[
\mathcal L_q=\frac{1}{N}\sum_{n=1}^{N}\sum_{j=1}^{9}
w_j(\widehat y_{n,j}^{(s)}-y_{n,j}^{(s)})^2.
\]

The in-plane heat-flux coefficients receive the largest weights. The purpose is not to make every coefficient error smaller. The scientific question is whether improving the \(\Gamma\) block produces a useful overall tradeoff.


In [ ]:
run_command([
    sys.executable, 'train_fp_closure.py',
    '--train', 'fp_train_data.npz',
    '--val', 'fp_validation_data.npz',
    '--out', 'fp_model_qweighted.npz',
    '--loss-mode', 'qweighted', '--q-weight', str(Q_WEIGHT),
    '--epochs', str(TRAIN_EPOCHS),
    '--hidden', '128', '--batch', '2048',
    '--seed', str(MODEL_SEED),
])


## 6. Compare both models on the complete validation condition

In [ ]:
run_command([
    sys.executable, 'evaluate_fp_closure.py',
    '--model', 'fp_model_uniform.npz', '--data', 'fp_validation_data.npz',
    '--label', 'uniform_validation', '--outdir', 'validation_eval',
])
run_command([
    sys.executable, 'evaluate_fp_closure.py',
    '--model', 'fp_model_qweighted.npz', '--data', 'fp_validation_data.npz',
    '--label', 'qweighted_validation', '--outdir', 'validation_eval',
])

u = pd.read_csv('validation_eval/uniform_validation_metrics.csv').set_index('group')
q = pd.read_csv('validation_eval/qweighted_validation_metrics.csv').set_index('group')
comparison = pd.DataFrame({
    'uniform_relL2': u['relative_L2'],
    'qweighted_relL2': q['relative_L2'],
})
comparison['qweighted_change_percent'] = 100*(comparison['qweighted_relL2']/comparison['uniform_relL2'] - 1)
display(comparison.loc[['all','C_block','Gamma_block','Gamma_block_high_q80']])


## Why the selection rule has two clauses

Improving only the `Gamma` block can be purchased by unacceptable degradation of the `C` block. The validation rule therefore requires a targeted improvement and a bounded collateral cost. Record the rule before generating blind data. Do not select a q-weight because its blind field plot looks better.


## 7. Freeze the model-selection rule

Use the following predeclared rule or write a different rule **before** opening blind results:

- choose the q-weighted model if it reduces validation `Gamma_block` relative \(L_2\) error;
- and its `C_block` error is no more than 20% worse than the uniform model;
- otherwise retain the uniform model and report that the tested q-weight did not produce an acceptable tradeoff.

A negative result is acceptable.


In [ ]:
gamma_improved = q.loc['Gamma_block','relative_L2'] < u.loc['Gamma_block','relative_L2']
C_within_tolerance = q.loc['C_block','relative_L2'] <= 1.20*u.loc['C_block','relative_L2']
SELECTED_MODEL = 'fp_model_qweighted.npz' if (gamma_improved and C_within_tolerance) else 'fp_model_uniform.npz'

selection_record = {
    'gamma_improved': bool(gamma_improved),
    'C_within_20_percent_tolerance': bool(C_within_tolerance),
    'selected_model': SELECTED_MODEL,
    'blind_lid_speed': BLIND_LID_SPEED,
    'settings_frozen_before_blind': True,
}
Path('fp_model_selection.json').write_text(json.dumps(selection_record, indent=2))
print(json.dumps(selection_record, indent=2))


# BLIND-TEST GATE

Do not continue until:

- both validation tables are saved;
- `fp_model_selection.json` exists;
- architecture, q-weight, seed, and stopping rule are frozen;
- you have written one sentence predicting what will happen at 800 m/s.


## 8. Generate and evaluate the blind coefficient condition

In [ ]:
run_command([
    sys.executable, 'generate_fp_cavity_dataset.py',
    '--out', 'fp_blind_coeff_data.npz', '--role', 'blind',
    '--lid-speeds', str(BLIND_LID_SPEED),
    '--density-scales', str(DENSITY_SCALE),
    '--seed', '301', *common_data_args,
])

for label, model in [('uniform_blind','fp_model_uniform.npz'),
                     ('qweighted_blind','fp_model_qweighted.npz')]:
    run_command([
        sys.executable, 'evaluate_fp_closure.py',
        '--model', model, '--data', 'fp_blind_coeff_data.npz',
        '--label', label, '--outdir', 'blind_eval',
    ])

ub = pd.read_csv('blind_eval/uniform_blind_metrics.csv').set_index('group')
qb = pd.read_csv('blind_eval/qweighted_blind_metrics.csv').set_index('group')
blind_comparison = pd.DataFrame({
    'uniform_relL2': ub['relative_L2'],
    'qweighted_relL2': qb['relative_L2'],
})
display(blind_comparison.loc[['all','C_block','Gamma_block','Gamma_block_high_q80']])


## A fair online timing experiment

Use identical physical configuration, seed policy, grid, particles, transient, sampling window, and output frequency. Synchronize GPU work before reading the clock. Report warm-up separately when relevant. Distinguish closure-kernel speedup from end-to-end solver speedup and include exact data-generation/training cost when discussing break-even.

The supplied reference path has two known savings: replacing the local exact solve and avoiding exact-path higher-moment work that is not needed by the learned path. State this explicitly when interpreting speedup.


## 9. Closed-loop blind cavity test

Coefficient regression is necessary but not sufficient. Small local coefficient errors can accumulate through thousands of particle updates. The selected model must therefore be inserted into the complete cavity solver and compared with an exact-FP run at the same blind condition.

The wrapper below uses the same seed, grid, particle budget, transient, and averaging window for both runs. It saves macroscopic fields, coefficient fields, and selected high-order non-equilibrium diagnostics.


In [ ]:
run_command([
    sys.executable, 'run_fp_cavity_test.py',
    '--model', SELECTED_MODEL,
    '--outdir', 'fp_closed_loop_blind',
    '--u-lid', str(BLIND_LID_SPEED),
    '--density-scale', str(DENSITY_SCALE),
    '--nx', str(TEST_NX), '--ny', str(TEST_NY),
    '--ppc', str(TEST_PPC),
    '--steps', str(TEST_STEPS),
    '--sample-start', str(TEST_SAMPLE_START),
    '--sample-stride', str(TEST_SAMPLE_STRIDE),
    '--seed', '2026',
])


## 10. Analyze the closed-loop result

In [ ]:
run_command([
    sys.executable, 'analyze_fp_cavity_results.py',
    '--physics', 'fp_closed_loop_blind/fp_cavity_PHYSICS.npz',
    '--ml', 'fp_closed_loop_blind/fp_cavity_ML.npz',
    '--outdir', 'fp_closed_loop_analysis',
])

metrics = pd.read_csv('fp_closed_loop_analysis/fp_cavity_metrics.csv')
display(metrics)
checks = json.loads(Path('fp_closed_loop_analysis/fp_physical_checks.json').read_text())
print(json.dumps(checks, indent=2))


In [ ]:
from IPython.display import Image, display
for name in ['fp_speed_comparison.png','fp_temperature_comparison.png',
             'fp_density_comparison.png','fp_centerlines.png']:
    display(Image(filename=f'fp_closed_loop_analysis/{name}'))


## Reading a closed-loop success or failure

Macroscopic agreement can coexist with early degradation of stress or heat flux. A stable simulation is not necessarily an accurate one. Conversely, modest local coefficient errors may be dynamically benign if the solver is insensitive in the tested regime.

Rank conclusions by strength: one reduced-budget blind case; repeated seeds at the same condition; multiple held-out speeds; unseen rarefaction; then grid/particle/time convergence. Do not generalize beyond the level actually tested.


## Required interpretation

Your report must distinguish the following layers of evidence:

1. **Validation coefficient error:** Does q-weighting improve \(\Gamma_i\) on the complete 500 m/s validation condition?
2. **Blind coefficient error:** Does the same tradeoff persist at 800 m/s?
3. **Closed-loop macroscopic error:** Are speed, temperature, density, pressure, and centerlines retained?
4. **High-order fidelity:** Are `q_norm`, `Rij_norm`, `Delta4_norm`, or `DM6_norm` more sensitive than the macroscopic fields?
5. **Stability and physics:** Are density and temperature positive? Is the upper flow driven in the lid direction? Is return flow present below?
6. **Cost:** What online speedup is measured for this reduced configuration? Do not call it a universal maximum.

A visually close contour is not enough. Use the CSV metrics and the physical checks.


## 11. Required failure or limitation analysis

At least one limitation must be documented. Acceptable examples include:

- q-weighting improves \(\Gamma\) but worsens \(C\) or macroscopic velocity;
- coefficient error is modest but high-order closed-loop diagnostics degrade;
- the 800 m/s condition lies too far from the training manifold;
- reduced particle counts make runtime or error comparisons noisy;
- the educational transient/averaging window is too short for a publication claim;
- low-order inputs are not universally identifiable for arbitrary non-equilibrium distributions.

Do not tune the model after examining the blind result.


## 12. Optional prize/research extension — unseen rarefaction

Only after completing the required speed study, test an unseen nominal \(Kn\approx0.5\) case by setting `density_scale=0.3` because

\[
Kn\approx0.15/0.3=0.5.
\]

This is a substantially harder test. The original training conditions are at nominal \(Kn\approx0.15\), so the result is a rarefaction stress test, not routine interpolation. Repeat the closed-loop comparison without retraining first. If it fails, use the failure to define the reliable scope of the closure. Any retraining extension must use new complete training conditions and a new untouched blind case.


In [ ]:
# OPTIONAL — run only after the required project is complete.
# run_command([
#     sys.executable, 'run_fp_cavity_test.py',
#     '--model', SELECTED_MODEL,
#     '--outdir', 'fp_closed_loop_Kn05',
#     '--u-lid', str(BLIND_LID_SPEED), '--density-scale', '0.3',
#     '--nx', str(TEST_NX), '--ny', str(TEST_NY), '--ppc', str(TEST_PPC),
#     '--steps', str(TEST_STEPS), '--sample-start', str(TEST_SAMPLE_START),
#     '--sample-stride', str(TEST_SAMPLE_STRIDE), '--seed', '3030',
# ])


## Final deliverables

Submit:

1. this fully executed notebook;
2. the exact train/validation/blind case table;
3. uniform and q-weighted validation/blind coefficient metrics;
4. `fp_model_selection.json` showing that selection occurred before blind testing;
5. selected exported model parameters;
6. closed-loop exact-FP and ML-FP NPZ outputs;
7. field/centerline figures and metric tables;
8. at least three physical checks;
9. one failure/tradeoff analysis;
10. a reproducibility and AI-assistance statement.

### Minimum code defense
You must be able to explain:

- how the 16 features are assembled;
- why the targets require the exact high-order closure;
- how output standardization and q-weighting interact;
- how the exported `W1...W5`, `b1...b5`, means, and scales reproduce the Keras forward pass in CuPy;
- why closed-loop testing is stronger than coefficient regression alone.


## Concept check and further reading

1. Why can low offline coefficient error fail in closed loop?
2. What does q-weighting change, and what does it not change?
3. Why are macroscopic and high-order metrics reported separately?
4. What must be synchronized for valid GPU timing?
5. What costs belong in an offline break-even analysis?
6. Which result would suggest feature non-observability rather than insufficient capacity?

Read Roohi (2026), *GPU-native neural surrogate for Fokker–Planck closure*, and the verification-oriented rarefied-ML references in `references/README.md`.


## Reproducibility record

Before closing the notebook, record:

- Python and package versions;
- dataset hash and helper versions;
- every physical case in development, validation, and blind sets;
- every seed and candidate value tried;
- the selection rule and when it was frozen;
- output filenames and units; and
- any cell that was skipped, changed, or run with a reduced budget.

Restart the kernel and run all cells in order. If the result changes materially, report the variability instead of selecting the preferred run.

## Troubleshooting without corrupting the experiment

| Symptom | Safe action | Unsafe action |
| --- | --- | --- |
| Missing helper/data file | Re-run the bootstrap and verify paths/hash | Download an unlabeled older copy |
| Training is slow | Use the documented smoke configuration, then label it “smoke” | Quietly reduce epochs/data in the final claim |
| Validation is poor | Inspect scaling, split, and baseline; revise on development data | Open the blind case to choose settings |
| Blind result fails | Report/localize failure and propose a new future experiment | Tune on the blind case while keeping its label “blind” |
| Stochastic result changes | Run multiple declared seeds and report mean/spread | Keep rerunning until one result looks good |
| A neural model loses to interpolation | Verify fairness, then recommend the simpler method | Hide the baseline |

## Final report outline

1. **Question and hypothesis** — one falsifiable sentence.
2. **Data and split** — physical cases, numerical source, and blind unit.
3. **Baseline** — simplest credible comparator using the same allowed information.
4. **Modification** — the one controlled change.
5. **Selection** — validation-only candidates and frozen rule.
6. **Blind numerical result** — aggregate errors and variability.
7. **Physical result** — at least two diagnostics tied to the flow.
8. **Failure or limitation** — where confidence ends.
9. **Cost and reproducibility** — runtime, environment, seeds, saved files.
10. **Conclusion** — helped, hurt, or revealed a tradeoff; no forced positive AI claim.


## Article-output contract

<!-- MIE690A article-aligned validation v3 -->

**Role:** Fokker--Planck closure fields and profile comparison.

All manuscript-facing figures must be generated from retained numerical/model outputs through the documented notebook or shared helper, saved under `results/`, and accompanied by machine-readable metrics. Do not redraw curves by eye or substitute a screenshot for a solver-to-reference comparison. The complete ownership table and exact output filenames are in [`ARTICLE_FIGURE_MAP.md`](../ARTICLE_FIGURE_MAP.md).
